### Model Training

This sections serves to use the prepared and cleaned dataset and use the model to train with the dataset.

In [56]:
## Load Dataset

import pandas as pd
import numpy as np
from dataset_class.job_post_dataset import JobPostingDataset
combined_df = pd.read_csv("./data/clean/fake_job_postings_ALL.csv")

#"benefits_len","benefits_word_count",

numeric_cols = ["telecommuting", "missing_count", "total_text_len", "company_profile_len", "description_len", 
                "requirements_len", "company_profile_word_count", "description_word_count", 
                "requirements_word_count", "salary_provided", "has_company_profile",
                "vague_location", "has_company_logo", "has_questions"]
non_binary_cols = [
    "missing_count",          # ← add this
    "total_text_len", 
    "company_profile_len", 
    "description_len", 
    "requirements_len", 
    #"benefits_len", 
    "company_profile_word_count", 
    "description_word_count", 
    "requirements_word_count", 
    #"benefits_word_count"
]
numerical_array = combined_df[numeric_cols].to_numpy(dtype=np.float32)  # (N, 16)
labels_array    = combined_df['fraudulent'].to_numpy(dtype=np.float32)  # (N,)

print(numerical_array.shape)
print(labels_array.shape)



(13485, 14)
(13485,)


In [57]:
from sklearn.model_selection import train_test_split

# 1. Split 80% Train, 20% "Rest" (temp_data)
train_data, temp_data = train_test_split(
    combined_df, test_size=0.2, random_state=42, stratify=combined_df['fraudulent']
)

# 2. Split that 20% into half (10% Val, 10% Test)
# FIX: Use temp_data['fraudulent'] for stratification
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, random_state=42, stratify=temp_data['fraudulent']
)

X_train_text = train_data['full_text'].tolist()
X_train_numeric = train_data[numeric_cols].values.astype(np.float32)
y_train = train_data['fraudulent'].values.tolist()

X_val_text = val_data['full_text'].tolist()
X_val_numeric = val_data[numeric_cols].values.astype(np.float32)
y_val = val_data['fraudulent'].values.tolist()

X_test_text = test_data['full_text'].tolist()
X_test_numeric = test_data[numeric_cols].values.astype(np.float32)
y_test = test_data['fraudulent'].values.tolist()

In [58]:
from sklearn.preprocessing import StandardScaler

non_binary_cols = [
    "missing_count",          # ← add this
    "total_text_len", 
    "company_profile_len", 
    "description_len", 
    "requirements_len", 
    #"benefits_len", 
    "company_profile_word_count", 
    "description_word_count", 
    "requirements_word_count", 
    #"benefits_word_count"
]

non_binary_indices = [numeric_cols.index(col) for col in non_binary_cols]

scaler = StandardScaler()

# Scale in-place, preserving original column order
X_train_numeric[:, non_binary_indices] = scaler.fit_transform(X_train_numeric[:, non_binary_indices])
X_val_numeric[:,   non_binary_indices] = scaler.transform(X_val_numeric[:,     non_binary_indices])
X_test_numeric[:,  non_binary_indices] = scaler.transform(X_test_numeric[:,    non_binary_indices])

# Verify
print(X_train_numeric.mean(axis=0).round(3))  # non-binary cols ≈ 0.0
print(X_train_numeric.std(axis=0).round(3))   # non-binary cols ≈ 1.0

[ 0.041 -0.     0.    -0.    -0.    -0.     0.     0.    -0.     0.155
  0.802  0.028  0.774  0.482]
[0.197 1.    1.    1.    1.    1.    1.    1.    1.    0.362 0.398 0.164
 0.418 0.5  ]


In [59]:
print(list(enumerate(numeric_cols)))  # see index → column name mapping
print(f"Col 0: {numeric_cols[0]}, mean: {X_train_numeric[:, 0].mean():.3f}")
print(f"Col 1: {numeric_cols[1]}, mean: {X_train_numeric[:, 1].mean():.3f}")

[(0, 'telecommuting'), (1, 'missing_count'), (2, 'total_text_len'), (3, 'company_profile_len'), (4, 'description_len'), (5, 'requirements_len'), (6, 'company_profile_word_count'), (7, 'description_word_count'), (8, 'requirements_word_count'), (9, 'salary_provided'), (10, 'has_company_profile'), (11, 'vague_location'), (12, 'has_company_logo'), (13, 'has_questions')]
Col 0: telecommuting, mean: 0.041
Col 1: missing_count, mean: -0.000


In [60]:
print(type(X_train_numeric))
print(X_train_numeric.shape)
print(X_train_numeric[0])

<class 'numpy.ndarray'>
(10788, 14)
[ 0.         -1.1116045   1.3279732   1.9723661  -0.29295725  1.283025
  2.2757115  -0.05722585  1.5487986   1.          1.          0.
  1.          1.        ]


In [61]:
# Verify all types before creating datasets
print(type(X_train_text),    type(X_train_text[0]))     # list, str
print(type(X_train_numeric), X_train_numeric.shape)     # ndarray, (N, 16)
print(type(y_train),         type(y_train[0]))          # list, int

<class 'list'> <class 'str'>
<class 'numpy.ndarray'> (10788, 14)
<class 'list'> <class 'int'>


In [62]:
print(X_train_numeric.mean(axis=0))   # should be close to 0
print(X_train_numeric.std(axis=0))    # should be close to 1

[ 4.0600669e-02 -3.5006956e-08  2.5636407e-08 -5.6576898e-08
 -4.7206349e-08 -4.2432671e-09  7.4257178e-09  7.4610782e-08
 -3.0940491e-08  1.5507972e-01  8.0209494e-01  2.7715981e-02
  7.7373010e-01  4.8183167e-01]
[0.19736326 1.         0.99999994 1.         0.99999994 1.
 1.         1.         1.         0.36198065 0.3984202  0.16415787
 0.41841587 0.49966982]


In [63]:
from gensim.models import FastText

fasttext_model_optimal = FastText.load("./optimal_fasttext.bin")

print(fasttext_model_optimal.wv.similarity('software', 'engineer'))
print(fasttext_model_optimal.wv.similarity('skills', 'experience'))
print(fasttext_model_optimal.wv.most_similar('water', topn=10))

0.43055043
0.51230246
[('saltwater', 0.8357968330383301), ('wastewater', 0.8045268654823303), ('backwater', 0.7969263195991516), ('stormwater', 0.7853497266769409), ('watering', 0.7785604596138), ('groundwater', 0.7723948955535889), ('whitewater', 0.7641539573669434), ('stillwater', 0.7530474662780762), ('waterloo', 0.729570746421814), ('deepwater', 0.7263585329055786)]


In [64]:
from nltk.tokenize import word_tokenize

train_tk_sentences = [word_tokenize(text.lower()) for text in X_train_text]  
val_tk_sentences = [word_tokenize(text.lower()) for text in X_val_text]  
test_tk_sentences = [word_tokenize(text.lower()) for text in X_test_text]  

In [65]:
print(train_tk_sentences[0])

['store', 'manager', 'papa', 'john', 's', 'pizza', 'gb', 'liv', 'take', 'out', 'brands', 'is', 'a', 'food', 'franchise', 'business', 'with', 'a', 'link', 'difference', 'being', 'we', 'want', 'you', 'to', 'genuinely', 'enjoy', 'your', 'workiing', 'experience', 'with', 'us', '.', 'we', 'hire', 'cheerful', ',', 'honest', 'and', 'hard', 'working', 'people', 'and', 'then', 'treat', 'them', 'well', 'offering', 'the', 'chance', 'to', 'learn', 'and', 'develop', 'wherever', 'possible', '.', 'with', '7', 'franchises', 'already', 'live', ',', 'this', 'young', 'business', 'is', 'run', 'by', 'people', 'you', 'can', 'trust', '.', 'our', 'focus', 'currently', 'is', 'on', 'papa', 'john', 's', 'one', 'of', 'the', 'largest', 'pizza', 'companies', 'in', 'the', 'world', ',', 'with', 'more', 'than', '4,300', 'stores', 'worldwide', 'delivering', 'better', 'ingredients', ',', 'better', 'pizza', '.', 'rapidly', 'growing', 'in', 'the', 'uk', ',', 'there', 'are', 'now', 'over', '200', 'papa', 'johns', 'outlets'

In [66]:
# Reserve 0 = <unk>, 1 = <pad>
vocab = {"<unk>": 0, "<pad>": 1}

# Start FastText words from index 2 onwards
for idx, word in enumerate(fasttext_model_optimal.wv.index_to_key):
    vocab[word] = idx + 2

print(f"Vocab size: {len(vocab)}")
print(f"<unk> index: {vocab['<unk>']}")  # 0
print(f"<pad> index: {vocab['<pad>']}")  # 1

def encode(tokenized_sentence, vocab):
    return [vocab.get(token, 0) for token in tokenized_sentence]  # 0 = <unk>

train_texts_tok = [encode(s, vocab) for s in train_tk_sentences]
val_texts_tok   = [encode(s, vocab) for s in val_tk_sentences]
test_texts_tok  = [encode(s, vocab) for s in test_tk_sentences]

Vocab size: 22096
<unk> index: 0
<pad> index: 1


In [67]:
from torch.utils.data import DataLoader

train_dataset = JobPostingDataset(
    texts_tok           = train_texts_tok,   # your tokenized sequences
    numerical_features  = X_train_numeric,
    labels              = y_train,
    max_len             = 2048
)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [68]:
val_dataset = JobPostingDataset(
    texts_tok           = val_texts_tok,   # your tokenized sequences
    numerical_features  = X_val_numeric,
    labels              = y_val,
    max_len             = 2048
)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=True)

In [69]:
test_dataset = JobPostingDataset(
    texts_tok           = test_texts_tok,   # your tokenized sequences
    numerical_features  = X_test_numeric,
    labels              = y_test,
    max_len             = 2048
)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True)

#### Testing Dataloader by sampling a batch

In [70]:
inputs,label = train_dataset[0]
print("input_ids shape:          ", inputs["input_ids"].shape)           # (2048,)
print("attention_mask shape:     ", inputs["attention_mask"].shape)      # (2048,)
print("numerical_features shape: ", inputs["numerical_features"].shape)  # (num_features,)
print("label:                    ", label)


input_ids shape:           torch.Size([2048])
attention_mask shape:      torch.Size([2048])
numerical_features shape:  torch.Size([14])
label:                     tensor(0.)


### Instantiating the model

In [71]:
import torch

vocab_size = len(vocab)
embed_dim  = fasttext_model_optimal.wv.vector_size

embedding_matrix = np.zeros((vocab_size, embed_dim), dtype=np.float32)
# index 0 (<unk>) stays zero
# index 1 (<pad>) stays zero

for word, idx in vocab.items():
    if word in fasttext_model_optimal.wv:
        embedding_matrix[idx] = fasttext_model_optimal.wv[word]

pretrained_embeddings = torch.tensor(embedding_matrix, dtype=torch.float)
print(pretrained_embeddings.shape)  # torch.Size([vocab_size, 100])

torch.Size([22096, 100])


In [72]:
from model_construction.model import FakeJobDetector

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FakeJobDetector(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    num_numerical_features=X_train_numeric.shape[1],
    pretrained_embeddings=pretrained_embeddings,
    gru_hidden_dim=64,
    num_hidden_dim=128,
    device=device,
)

print(model)

FakeJobDetector(
  (embedding): Embedding(22096, 100, padding_idx=1)
  (bigru): BiGRUBlock(
    (gru): GRU(100, 64, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  )
  (attention): MultiHeadAttentionPooling(
    (heads): ModuleList(
      (0-1): 2 x Linear(in_features=128, out_features=1, bias=True)
    )
    (projection): Linear(in_features=256, out_features=128, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (numerical): NumericalBlock(
    (layers): Sequential(
      (0): Linear(in_features=14, out_features=128, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=128, out_features=128, bias=True)
      (4): ReLU()
      (5): Dropout(p=0.3, inplace=False)
    )
  )
  (classifier): Linear(in_features=256, out_features=1, bias=True)
)


In [73]:
def audit_numerical_features(dataloader, feature_names=None):
        import numpy as np

        all_features = []
        all_labels   = []

        for inputs, targets in dataloader:
            all_features.append(inputs['numerical_features'].numpy())
            all_labels.append(targets.numpy())

        X = np.vstack(all_features)   # (N, num_features)
        y = np.concatenate(all_labels)

        print(f"Shape: {X.shape}  |  Fake rate: {y.mean():.3f}\n")

        names = feature_names or [f"feat_{i}" for i in range(X.shape[1])]

        issues = []
        for i, name in enumerate(names):
            col      = X[:, i]
            variance = col.var()
            nan_pct  = np.isnan(col).mean() * 100
            scale    = np.abs(col).max()

            fake_mean = col[y == 1].mean() if (y == 1).any() else float('nan')
            real_mean = col[y == 0].mean() if (y == 0).any() else float('nan')
            sep       = abs(fake_mean - real_mean) / (col.std() + 1e-8)  # Cohen's d approx

            flag = []
            if variance < 1e-4:      flag.append("NEAR-ZERO VARIANCE")
            if nan_pct > 0:          flag.append(f"{nan_pct:.1f}% NaN")
            if scale > 100:          flag.append(f"LARGE SCALE (max={scale:.0f}) — needs normalisation")
            if sep < 0.1:            flag.append("LOW SEPARABILITY (Cohen's d < 0.1)")

            status = " | ".join(flag) if flag else "ok"
            print(f"  {name:<30} var={variance:8.4f}  sep={sep:.3f}  {status}")
            if flag:
                issues.append(name)

        print(f"\n{len(issues)}/{len(names)} features flagged")
        return issues

In [74]:
issues = audit_numerical_features(test_dataloader)

Shape: (1349, 14)  |  Fake rate: 0.051

  feat_0                         var=  0.0445  sep=0.273  ok
  feat_1                         var=  1.0482  sep=0.348  ok
  feat_2                         var=  1.0577  sep=0.696  ok
  feat_3                         var=  1.0076  sep=0.808  ok
  feat_4                         var=  1.0459  sep=0.289  ok
  feat_5                         var=  0.9711  sep=0.515  ok
  feat_6                         var=  1.0260  sep=0.823  ok
  feat_7                         var=  1.0568  sep=0.286  ok
  feat_8                         var=  0.9859  sep=0.511  ok
  feat_9                         var=  0.1294  sep=0.487  ok
  feat_10                        var=  0.1605  sep=1.378  ok
  feat_11                        var=  0.0336  sep=0.216  ok
  feat_12                        var=  0.1774  sep=1.273  ok
  feat_13                        var=  0.2491  sep=0.535  ok

0/14 features flagged


In [79]:
train_losses, val_losses = model.fit(
    dataloader     = train_dataloader,
    val_dataloader = val_dataloader,
    num_epochs     = 20,
    learning_rate  = 1e-3,
    save_path      = "best_model.pt",
)

# Evaluate with lower threshold to catch more fakes
model.evaluate(test_dataloader, threshold=0.3)



Epoch 1/20 | Train Loss: 0.0171 | Val Loss: 0.0170
  ✅ Best model saved (val_loss=0.0170)
Epoch 2/20 | Train Loss: 0.0137 | Val Loss: 0.0183
  ⚠️ No improvement (best_val_loss=0.0170)
Epoch 3/20 | Train Loss: 0.0135 | Val Loss: 0.0198
  ⚠️ No improvement (best_val_loss=0.0170)
Epoch 4/20 | Train Loss: 0.0117 | Val Loss: 0.0169
  ✅ Best model saved (val_loss=0.0169)
Epoch 5/20 | Train Loss: 0.0121 | Val Loss: 0.0234
  ⚠️ No improvement (best_val_loss=0.0169)
Epoch 6/20 | Train Loss: 0.0115 | Val Loss: 0.0149
  ✅ Best model saved (val_loss=0.0149)
Epoch 7/20 | Train Loss: 0.0096 | Val Loss: 0.0241
  ⚠️ No improvement (best_val_loss=0.0149)
Epoch 8/20 | Train Loss: 0.0120 | Val Loss: 0.0228
  ⚠️ No improvement (best_val_loss=0.0149)
Epoch 9/20 | Train Loss: 0.0125 | Val Loss: 0.0188
  ⚠️ No improvement (best_val_loss=0.0149)
Epoch 10/20 | Train Loss: 0.0107 | Val Loss: 0.0249
  ⚠️ No improvement (best_val_loss=0.0149)
Epoch 11/20 | Train Loss: 0.0123 | Val Loss: 0.0222
  ⚠️ No improvement

In [ ]:
model.evaluate_branches(test_dataloader, threshold=0.3)

              precision    recall  f1-score   support

        Real       0.99      0.95      0.97      1280
        Fake       0.47      0.87      0.61        69

    accuracy                           0.94      1349
   macro avg       0.73      0.91      0.79      1349
weighted avg       0.97      0.94      0.95      1349


Full model
              precision    recall  f1-score   support

        Real       0.99      0.95      0.97      1280
        Fake       0.47      0.87      0.61        69

    accuracy                           0.94      1349
   macro avg       0.73      0.91      0.79      1349
weighted avg       0.97      0.94      0.95      1349


NLP only
              precision    recall  f1-score   support

        Real       0.99      0.91      0.95      1280
        Fake       0.36      0.90      0.52        69

    accuracy                           0.91      1349
   macro avg       0.68      0.91      0.73      1349
weighted avg       0.96      0.91      0.93      134

c:\Users\Iris\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Iris\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Iris\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



Both zeroed
              precision    recall  f1-score   support

        Real       0.00      0.00      0.00      1280
        Fake       0.05      1.00      0.10        69

    accuracy                           0.05      1349
   macro avg       0.03      0.50      0.05      1349
weighted avg       0.00      0.05      0.00      1349



c:\Users\Iris\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Iris\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Iris\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

        Real       0.99      0.97      0.98      1280
        Fake       0.60      0.86      0.70        69

    accuracy                           0.96      1349
   macro avg       0.79      0.91      0.84      1349
weighted avg       0.97      0.96      0.97      1349


Full model
              precision    recall  f1-score   support

        Real       0.99      0.97      0.98      1280
        Fake       0.60      0.86      0.70        69

    accuracy                           0.96      1349
   macro avg       0.79      0.91      0.84      1349
weighted avg       0.97      0.96      0.97      1349


NLP only
              precision    recall  f1-score   support

        Real       0.99      0.96      0.97      1280
        Fake       0.52      0.86      0.65        69

    accuracy                           0.95      1349
   macro avg       0.76      0.91      0.81      1349
weighted avg       0.97      0.95      0.96      134

...

c:\Users\Iris\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Iris\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Iris\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

        Real       0.99      0.99      0.99      1280
        Fake       0.82      0.80      0.81        69

    accuracy                           0.98      1349
   macro avg       0.90      0.89      0.90      1349
weighted avg       0.98      0.98      0.98      1349


Full model
              precision    recall  f1-score   support

        Real       0.99      0.99      0.99      1280
        Fake       0.82      0.80      0.81        69

    accuracy                           0.98      1349
   macro avg       0.90      0.89      0.90      1349
weighted avg       0.98      0.98      0.98      1349


NLP only
              precision    recall  f1-score   support

        Real       0.99      0.98      0.99      1280
        Fake       0.74      0.83      0.78        69

    accuracy                           0.98      1349
   macro avg       0.87      0.91      0.88      1349
weighted avg       0.98      0.98      0.98      134

c:\Users\Iris\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Iris\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Iris\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

        Real       0.99      0.99      0.99      1280
        Fake       0.89      0.78      0.83        69

    accuracy                           0.98      1349
   macro avg       0.94      0.89      0.91      1349
weighted avg       0.98      0.98      0.98      1349


Full model
              precision    recall  f1-score   support

        Real       0.99      0.99      0.99      1280
        Fake       0.89      0.78      0.83        69

    accuracy                           0.98      1349
   macro avg       0.94      0.89      0.91      1349
weighted avg       0.98      0.98      0.98      1349


NLP only
              precision    recall  f1-score   support

        Real       0.99      0.99      0.99      1280
        Fake       0.87      0.78      0.82        69

    accuracy                           0.98      1349
   macro avg       0.93      0.89      0.91      1349
weighted avg       0.98      0.98      0.98      134

c:\Users\Iris\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Iris\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Iris\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [82]:
best_model = FakeJobDetector(
    vocab_size             = vocab_size,
    embed_dim              = embed_dim,
    gru_hidden_dim         = 64,
    num_numerical_features = 14,
    num_hidden_dim         = 128,
    dropout                = 0.3,
    pretrained_embeddings  = pretrained_embeddings,
    device                 = device
)

best_model.load("best_model.pt")

In [83]:
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    print(f"\n--- Threshold: {threshold} ---")
    best_model.evaluate(test_dataloader, threshold=threshold)


--- Threshold: 0.3 ---
              precision    recall  f1-score   support

        Real       0.99      0.97      0.98      1280
        Fake       0.64      0.84      0.73        69

    accuracy                           0.97      1349
   macro avg       0.82      0.91      0.86      1349
weighted avg       0.97      0.97      0.97      1349


--- Threshold: 0.4 ---
              precision    recall  f1-score   support

        Real       0.99      0.98      0.98      1280
        Fake       0.67      0.83      0.74        69

    accuracy                           0.97      1349
   macro avg       0.83      0.90      0.86      1349
weighted avg       0.97      0.97      0.97      1349


--- Threshold: 0.5 ---
              precision    recall  f1-score   support

        Real       0.99      0.99      0.99      1280
        Fake       0.79      0.81      0.80        69

    accuracy                           0.98      1349
   macro avg       0.89      0.90      0.89      1349
we